# Mock Experiment v0.9 — Identifiability of a Temporal Residual
Synthetic inference only. No real experimental data are used. Goal: determine when epsilon can or cannot be distinguished from ordinary nuisance shifts.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(7); t=np.linspace(0,10,1001)
def sigmoid(x): return 1/(1+np.exp(-x))
window=sigmoid(7*(t-3))*sigmoid(7*(7-t))
dwindow=np.gradient(window,t)
# temporal signal template: deliberately contains state-like + transition-localized pieces
F=.35*window+.65*np.tanh(1.5*dwindow)
# conventional nuisance templates
G=window                         # interaction-induced plateau shift
D=np.exp(-.5*((t-5)/2.2)**2)    # slow thermal/drift-like shift
C=np.ones_like(t)                # offset/common calibration
eps_true=2.0e-4; beta_true=1.4e-4; drift_true=-8e-5; offset_true=3e-5; sigma=5e-5
y=eps_true*F+beta_true*G+drift_true*D+offset_true*C+rng.normal(0,sigma,len(t))
plt.figure(figsize=(9,4.5)); plt.plot(t,y,label='synthetic residual',alpha=.7); plt.plot(t,eps_true*F,label='injected temporal component'); plt.legend(); plt.xlabel('t'); plt.grid(alpha=.25); plt.show()


## Linear inference helper

In [ ]:
def fit(cols,y,sigma):
    X=np.column_stack(cols); cov=np.linalg.inv(X.T@X)*(sigma**2); bhat=np.linalg.solve(X.T@X,X.T@y); se=np.sqrt(np.diag(cov)); return bhat,se,cov


## Wrong model: fit epsilon but omit interaction nuisance
This intentionally demonstrates false attribution.

In [ ]:
b,se,_=fit([F,D,C],y,sigma); print('wrong-model eps=',b[0],'+/-',se[0],' true=',eps_true)


## Full model: temporal + interaction + drift + offset

In [ ]:
b,se,cov=fit([F,G,D,C],y,sigma); names=['epsilon','interaction','drift','offset']
for n,x,s in zip(names,b,se): print(n,x,'+/-',s,' z=',x/s)
corr=cov/np.sqrt(np.outer(np.diag(cov),np.diag(cov))); print('parameter correlation matrix\n',np.round(corr,3))


## Degeneracy test
If the proposed temporal template is identical to an ordinary interaction-shift template, epsilon is mathematically non-identifiable.

In [ ]:
Xdeg=np.column_stack([G,G,D,C]); print('degenerate design rank=',np.linalg.matrix_rank(Xdeg),' of ',Xdeg.shape[1]); print('condition=',np.linalg.cond(Xdeg))


## Transition-only template improves orthogonality
A localized derivative-like signature can be less degenerate with a plateau-like interaction shift.

In [ ]:
T=np.tanh(1.5*dwindow)
def cosine(a,b): return np.dot(a,b)/np.sqrt(np.dot(a,a)*np.dot(b,b))
print('cos(F,G)=',cosine(F,G)); print('cos(T,G)=',cosine(T,G))


## Control channel
A control clock receives the same nuisance terms but no injected temporal-identity signal. Difference channels can suppress common systematics.

In [ ]:
yc=beta_true*G+drift_true*D+offset_true*C+rng.normal(0,sigma,len(t))
ydiff=y-yc
sdiff=np.sqrt(2)*sigma
b,se,_=fit([F,C],ydiff,sdiff); print('control-difference eps=',b[0],'+/-',se[0],' true=',eps_true)


## Detection-threshold scan

In [ ]:
sigmas=np.logspace(-6,-3,80); z=[]
X=np.column_stack([F,G,D,C]); inv=np.linalg.inv(X.T@X)
for s in sigmas: z.append(abs(eps_true)/(s*np.sqrt(inv[0,0])))
plt.figure(figsize=(8,4.5)); plt.loglog(sigmas,z); plt.axhline(5,ls='--',label='5 sigma'); plt.xlabel('per-sample noise sigma'); plt.ylabel('expected epsilon significance'); plt.legend(); plt.grid(alpha=.25); plt.show()


## Interpretation
Identifiability depends on template geometry, not only signal amplitude. If F lies in the span of conventional nuisance templates, no amount of lower random noise separates epsilon without external information. A useful temporal hypothesis must therefore predict a distinctive cross-system or transition-dependent pattern, not merely an arbitrary clock-frequency shift.